In [1]:
import os
import json
import re
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

## Opening the file

In [ ]:
with open("full_question_solution_latest.json", "r") as f:
    content = json.load(f)

# Print header
print(f"{'Filename':<30} {'Directness Level':<15} {'      Last Teacher Question'}")
print("-" * 100)

# For each file in the content
for filename, data in content.items():
    # Get the directness levels
    levels = [level for level in data.keys() if level != 'solution']
    
    # For each directness level
    for level in sorted(levels):  # Sort to ensure consistent order (least, more, most)
        # Extract the conversation
        conversation = data[level]
        
        # Extract the last teacher question
        # This assumes the conversation format has the teacher's questions marked in some way
        teacher_questions = re.findall(r"Teacher: (.*?)(?=Student:|$)", conversation, re.DOTALL)
        last_question = teacher_questions[-1].strip() if teacher_questions else "No question found"
        
        # Truncate long questions for display
        if len(last_question) > 70:
            last_question = last_question[:67] + "..."
        print(f"{filename:<30} {level:<15} {last_question}")
    
    print("-" * 100)

## Import the tokenizer and the model

In [ ]:
# Selecting the model
# NOTE: CodeLlama requires license. You must obtain one thorugh Meta and registered it in your huggingface account.
model_name = "Qwen/Qwen2.5-Coder-14B-Instruct"
# model_name = "meta-llama/CodeLlama-13b-Instruct-hf"

# Load the tokenizer and model.
cache_custom_dir = "/data/gpfs/projects/punim2402/huggingface"
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir = cache_custom_dir)
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_custom_dir, device_map="cuda")

# set the model to evaluation mode
model.eval()

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 5120)
    (layers): ModuleList(
      (0-47): 48 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=5120, out_features=5120, bias=True)
          (k_proj): Linear(in_features=5120, out_features=1024, bias=True)
          (v_proj): Linear(in_features=5120, out_features=1024, bias=True)
          (o_proj): Linear(in_features=5120, out_features=5120, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=5120, out_features=13824, bias=False)
          (up_proj): Linear(in_features=5120, out_features=13824, bias=False)
          (down_proj): Linear(in_features=13824, out_features=5120, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((5120,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((5120,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((5120,), eps=1e-06)
    (rotary_emb

## Main function for the Log Prob Calculation

In [ ]:
# Main function for the log prob
def compute_log_prob(context_question: str, reference_answer: str) -> float:
    """
    Compute the total log probability of the expected_answer given the question.
    """
    system_prompt = """You are a beginner programming student. Based on your buggy code, you are having a conversation with a teacher 
    that is helping you solve the problem. The teacher is a Socratic Tutor that is leading you to the correct answer with varying level
    of question directness. Based on the final question, do your best to think of the solution to the problem. Provide a response to the 
    final question from the teacher."""

    context = """You are a beginner programming student, and you are trying to debug a piece of code. You are having a conversation with a teacher who is guiding you 
    through the debugging process. The teacher follows the Socratic method, asking questions that lead you to the correct solution. 
    Your teacher's questions vary in directness, ranging from questions that directly point out the error in your code to more exploratory, 
    indirect questions that help you think about concepts or related issues.
    Based on the final question from the teacher, think about the solution to the problem and provide a response to the question."""
    
    # Tokenize the question and answer.
    messages_with_answer = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": context + context_question},
        {"role": "assistant", "content": "Student: "  + reference_answer}
    ]
    input_ids = tokenizer.apply_chat_template(messages_with_answer, return_tensors="pt").to("cuda")

    # Extract the user message length
    messages_to_last_utterance = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": context + context_question}
    ]
    input_ids_up_to_user = tokenizer.apply_chat_template(messages_to_last_utterance, return_tensors="pt")
    user_length = input_ids_up_to_user.shape[1]

    # Extract the answer token ids
    answer_token_ids = input_ids[:, user_length:]
    answer_length = answer_token_ids.shape[1]
    
    # Calculate the log probabilities of the answer tokens
    with torch.no_grad():
        outputs = model(input_ids)
        logits = outputs.logits
    
    # Slice logits to get those corresponding to the answer tokens
    logits_for_answer = logits[:, user_length - 1 : user_length + answer_length - 1, :]
    
    # Compute log probabilities
    log_probs = F.log_softmax(logits_for_answer, dim=-1)
    token_log_probs = log_probs.gather(2, answer_token_ids.unsqueeze(-1)).squeeze(-1)
    total_log_prob = token_log_probs.sum().item()
    
    return total_log_prob

## Running the calculation for each question and solution pair

In [ ]:
# Compute and print the log probability for each pair.
results = {}
for filename, value in content.items():
    # print(filename)
    results[filename] = {}
    reference_answer = value['solution']

    # Retrieve the levels
    levels = list(value.keys())
    levels.remove('solution')
    
    for level in levels:
        context_question = value[level]
        log_prob = compute_log_prob(context_question, reference_answer)
        results[filename][level] = log_prob

0_1_fibonacci
0_2_fibonacci
0_5_fibonacci
0_6_fibonacci
10_39_xnglxsh
11_40_palindrome
12_41_reversing_a_list
13_42_limit
14_43_used_twice
15_44_sequential_search
15_45_sequential_search
16_46_substring_length
16_56_substring_length
17_47_topk_socratic
18_48_password_validator
19_49_word_counter
19_50_word_counter
1_10_calculating_a_grade
1_11_calculating_a_grade
1_13_calculating _a_grade
1_8_calculating_a_grade
1_9_calculating_a_grade
20_51_spell_checker
21_52_fahrenheit_to_celsius
22_53_cookie_purchase
24_29_factorial_socratic
25_55_insert_to_linked_list
2_18_splitting_cookies
3_20_counting_down
4_22_removing_even_number
4_23_removing_even_numer
4_25_removing_even_numer
4_26_removing_even_numbers
4_28_removing_even_numbers
56_15_compute_average
58_58_splitting_apples
58_59_splitting_apples
59_60_product
5_30_sorted_words
60_61_largest_number
61_62_is_even
62_63_summing_between_integers
63_64_good_dinner
64_65_count_ones
65_66_list_range
66_67_last_index_of
66_68_last_index_of
66_69_l

## Saving output to file

In [ ]:
# Change name accordingly to the model used
with open("log_prob_score_qwen.json", "w") as f:
    json.dump(results, f)

## Opening Result File 

In [13]:
for key, values in results.items():
    for score_type in values:
        values[score_type] = abs(values[score_type])

def compare(a, b):
    return 1 if a > b else 0

def compute_accuracy(data):
    # Result dictionary to store comparison results
    result = {}

    # Counters for total and correct comparisons
    least_more = []
    more_most = []
    least_most = []
    total_comparisons = 0


    # Iterate through the data
    for key, values in data.items():
        comparisons = {}
        # check = 0

        # Perform comparisons if the keys exist
        if 'least' in values and 'more' in values:
            res = compare(values['least'], values['more'])
            comparisons['least_vs_more'] = res
            total_comparisons += 1
            least_more.append(res)

            # if not res:
            #     check += 1
            

        if 'more' in values and 'most' in values:
            res = compare(values['more'], values['most'])
            comparisons['more_vs_most'] = res
            total_comparisons += 1
            more_most.append(res)


            # if not res:
            #     check += 1

        if 'least' in values and 'most' in values:
            # if check % 2 == 0:
            #     continue
            res = compare(values['least'], values['most'])
            comparisons['least_vs_most'] = res
            total_comparisons += 1
            least_most.append(res)
        
        correct_comparisons = sum(least_more) + sum(more_most) + sum(least_most)
        result[key] = comparisons

    # Calculate percentage of correct comparisons
    accuracy_percentage = (correct_comparisons / total_comparisons) * 100 if total_comparisons > 0 else 0

    # Display the results
    # print("Comparison Results:", result)
    print("\n")
    print("Unique conversation:", len(data))
    print("Total Comparisons:", total_comparisons, "\n")

    print(f"Overrall Accuracy: {accuracy_percentage:.2f}% ({correct_comparisons}/{total_comparisons} correct)")
    print("Least vs More Accuracy: ", sum(least_more)/len(least_more))
    print("More vs Most Accuracy: ", sum(more_most)/len(more_most))
    print("Least vs Most Accuracy: ", sum(least_most)/len(least_most))
    print("\n")

compute_accuracy(results)



Unique conversation: 56
Total Comparisons: 162 

Overrall Accuracy: 54.32% (88/162 correct)
Least vs More Accuracy:  0.6226415094339622
More vs Most Accuracy:  0.4339622641509434
Least vs Most Accuracy:  0.5714285714285714




In [11]:
import pingouin
def perform_wilcoxon_test(data):
    # Collect "least" and "most" scores
    least_scores = []
    more_scores = []
    most_scores = []

    for key, values in data.items():
        if 'least' in values and 'most' in values:
            least_scores.append(values['least'])
            most_scores.append(values['most'])
            
        if 'least' in values and 'more' in values:
            more_scores.append(values['more'])
 
    # Perform Wilcoxon Signed-Rank Test
    print("Least with More Test:")
    print("-" * 70)
    stat_more = pingouin.wilcoxon(more_scores, least_scores, alternative='less')
    print(stat_more)

    print("\n" * 3)

    print("Least with Most Test:")
    print("-" * 70)        
    stat_most = pingouin.wilcoxon(most_scores, least_scores, alternative='less')
    print(stat_most)

    

perform_wilcoxon_test(results)

Least with More Test:
----------------------------------------------------------------------
          W-val alternative     p-val       RBC      CLES
Wilcoxon  684.0        less  0.177267 -0.142857  0.496173




Least with Most Test:
----------------------------------------------------------------------
          W-val alternative    p-val       RBC     CLES
Wilcoxon  294.0        less  0.00002 -0.631579  0.53986


In [10]:
def sort_by_least_most_difference(data):
    # Create list of tuples (key, difference)
    diff_pairs = []
    
    for key, values in data.items():
        if 'least' in values and 'most' in values:
            diff = abs(values['least']) - abs(values['most'])
            diff_pairs.append((key, diff, values['least'], values['most']))
    
    # Sort by difference (smallest to largest)
    sorted_pairs = sorted(diff_pairs, key=lambda x: x[1])
    
    # Print sorted results
    print("\nExamples sorted by least-most difference (smallest to largest):")
    print("-" * 70)
    print(f"{'Key':<30} {'Least':<10} {'Most':<10} {'Difference':<10}")
    print("-" * 70)
    
    for key, diff, least, most in sorted_pairs:
        print(f"{key:<30} {least:<10.5f} {most:<10.5f} {diff:<10.5f}")
    
    # Return sorted keys in case you need them
    return [pair[0] for pair in sorted_pairs]

# Call the function after your existing code
sorted_keys = sort_by_least_most_difference(results)


Examples sorted by least-most difference (smallest to largest):
----------------------------------------------------------------------
Key                            Least      Most       Difference
----------------------------------------------------------------------
59_60_product                  105.52400  113.93790  -8.41389  
0_5_fibonacci                  104.51492  110.83760  -6.32268  
13_42_limit                    105.37536  111.50693  -6.13157  
64_65_count_ones               141.34564  144.43439  -3.08875  
19_49_word_counter             153.12143  156.12665  -3.00522  
12_41_reversing_a_list         104.97899  107.81619  -2.83720  
24_29_factorial_socratic       206.04993  208.16090  -2.11098  
58_58_splitting_apples         92.35085   94.28395   -1.93310  
3_20_counting_down             149.69902  151.62448  -1.92546  
1_9_calculating_a_grade        159.00446  160.81676  -1.81230  
74_77_disney_vacation_club     143.87915  144.73468  -0.85553  
7_35_integer_grouping    